In [ ]:
import os
import numpy as np
import mlflow

### Load the data

Download the artifacts from the MLflow artifact registry. This expects MLFLOW_TRACKING_URI to have been setup as an environment variable *before* launching this notebook.

In [ ]:
mlflow.set_experiment("era5_drift_monitor_s3")
runs = mlflow.search_runs(filter_string="params.reference_years = '1980s' AND params.recent_year = '2025'", order_by=["start_time DESC"])
run_id = runs.iloc[0]["run_id"]
print("nruns:", len(runs))
print("Oldest run:")
runs.iloc[0]

In [ ]:
client = mlflow.MlflowClient()
for f in client.list_artifacts(run_id):
    print(f.path, f.is_dir, f.file_size)

In [ ]:
path = mlflow.artifacts.download_artifacts(run_id=run_id, dst_path="./downloaded_artifacts")

In [ ]:
ac_ref_field = np.load("downloaded_artifacts/ac_ref_field.npy")
ac_change_field = np.load("downloaded_artifacts/ac_change_field.npy")
rmse_frozen_field = np.load("downloaded_artifacts/rmse_frozen_field.npy")
rmse_recent_field = np.load("downloaded_artifacts/rmse_recent_field.npy")
drift_field = np.load("downloaded_artifacts/drift_field.npy")

reg_coeff_ct_field = np.load("downloaded_artifacts/reg_coeff_ct_field.npy")
reg_coeff_dmt_field = np.load("downloaded_artifacts/reg_coeff_dmt_field.npy")
reg_coeff_dzt_field = np.load("downloaded_artifacts/reg_coeff_dzt_field.npy")


In [ ]:
print(ac_ref_field.shape)
print(ac_change_field.shape)
print(drift_field.shape)

print(reg_coeff_ct_field.shape)
print(reg_coeff_dmt_field.shape)
print(reg_coeff_dzt_field.shape)


### Linear Regression results

Plot the autocorrelation, first coefficient and gradient coefficients of the linear regression

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
# Western Europe, 1deg grid
N, W, S, E = 60, -10, 40, 20
lon_min, lon_max, lon_deg = W, E, 1
lat_min, lat_max, lat_deg = S, N, 1

lon = np.linspace(lon_min, lon_max, int((lon_max-lon_min)/lon_deg+1))
lat = np.linspace(lat_min, lat_max, int((lat_max-lat_min)/lat_deg+1))
LON, LAT = np.meshgrid(lon, lat)

lon_red = lon[1:-1]
lat_red = lat[1:-1]
LON_red, LAT_red = np.meshgrid(lon_red, lat_red)

In [ ]:
def PlotOverWesternEurope(z, z_label, LON = LON, LAT = LAT, cmap="viridis", title="Atmospheric measurements over Western Europe"):
    fig, ax = plt.subplots(figsize=(12, 10.5), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent([LON.min(), LON.max(), LAT.min(), LAT.max()], crs=ccrs.PlateCarree())
    
    mesh = ax.pcolormesh(LON, LAT, z, transform=ccrs.PlateCarree(), cmap=cmap, shading="auto")
    # mesh = ax.pcolormesh(LON, LAT, z, transform=ccrs.PlateCarree(), norm=colors.CenteredNorm(), cmap=cmap, shading="auto")
    
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
    # ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", zorder=0)
    gl = ax.gridlines(draw_labels=["bottom", "left"], linewidth=0.4, alpha=0.5)
    
    plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7, label=z_label)
    plt.title(title)
    plt.show()

In [ ]:
def PlotArrowsOverWesternEurope(z_x, z_y, z_label, LON = LON, LAT = LAT, plottype="quiver", title="Atmospheric measurements over Western Europe"):

    fig, ax = plt.subplots(figsize=(9, 8), subplot_kw={"projection": ccrs.PlateCarree()})
    ax.set_extent([LON.min(), LON.max(), LAT.min(), LAT.max()], crs=ccrs.PlateCarree())

    if plottype == "quiver":
        q = ax.quiver(LON, LAT, z_x, z_y,
                      transform=ccrs.PlateCarree(),
                      scale=60,                # larger = shorter arrows
                      width=0.002,
                     )
    elif plottype == "stream":
        q = ax.streamplot(LON, LAT, z_x, z_y,
                      transform=ccrs.PlateCarree()
                     )
    else:
        print("plottype not recognized")
        return
    

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, linestyle=":")
    # ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", zorder=0)
    gl = ax.gridlines(draw_labels=["bottom", "left"], linewidth=0.4, alpha=0.5)
    
    plt.title(title)
    plt.show()

In [ ]:
PlotOverWesternEurope(ac_ref_field, "MSLP auto-correlation (in training years) [-]", LON, LAT)

In [ ]:
PlotOverWesternEurope(reg_coeff_ct_field, "Linear regression coefficient on central MSLP [-]", LON_red, LAT_red)

In [ ]:
PlotArrowsOverWesternEurope(-reg_coeff_dzt_field, -reg_coeff_dmt_field, "Linear response to gradient on MSLP [-]", LON_red, LAT_red, title="Linear regression on ERA5 pressure data: Pressure evolution over Western Europe")#, plottype="stream")

### Performance

Compare RMSE of either the frozen or recent model applied on recent data

In [ ]:
PlotOverWesternEurope(rmse_recent_field, "RMSE from recent year to itself [Pa]", LON_red, LAT_red)

In [ ]:
PlotOverWesternEurope(drift_field, "RMSE drift percentage [-]", LON_red, LAT_red, cmap = "Reds", title="Relative drift of linear regression model on ERA5 pressure from the 80s to 2025" )